In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.plotting import plot_decision_regions
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
!pip install lightgbm -q

import lightgbm as lgb
from lightgbm import LGBMRegressor  # or LGBMClassifier for classification


In [3]:
df = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
df.head(2)

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7


In [4]:
df_1 = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')
df_1.shape

(270000, 12)

In [5]:
df.shape

(630000, 13)

In [6]:
df.info(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                630000 non-null  int64  
 1   age               630000 non-null  int64  
 2   gender            630000 non-null  object 
 3   course            630000 non-null  object 
 4   study_hours       630000 non-null  float64
 5   class_attendance  630000 non-null  float64
 6   internet_access   630000 non-null  object 
 7   sleep_hours       630000 non-null  float64
 8   sleep_quality     630000 non-null  object 
 9   study_method      630000 non-null  object 
 10  facility_rating   630000 non-null  object 
 11  exam_difficulty   630000 non-null  object 
 12  exam_score        630000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 62.5+ MB


In [7]:
df.isnull().sum()

id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [8]:
df.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
629995    False
629996    False
629997    False
629998    False
629999    False
Length: 630000, dtype: bool

In [9]:
test = df_1.copy()
test_ids = test["id"]
test = test.drop(columns=["id"])
test = test.drop(columns=["gender", "internet_access", "course", "exam_difficulty"])

In [10]:
df['sleep_quality'].value_counts()

sleep_quality
poor       213675
good       213089
average    203236
Name: count, dtype: int64

In [11]:
df['study_method'].value_counts()

study_method
coaching         131697
self-study       131131
mixed            123086
group study      123009
online videos    121077
Name: count, dtype: int64

In [12]:
df['facility_rating'].value_counts()

facility_rating
medium    214082
low       212378
high      203540
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

target_col = "exam_score"
y = df[target_col]
X = df.drop(columns=[target_col])

In [14]:
from sklearn.preprocessing import LabelEncoder

drop_cols = ["id", "gender", "internet_access", "course", "exam_difficulty"]

X = df.drop(columns=drop_cols + ["exam_score"])
y = df["exam_score"]


X_encoded = X.copy()
label_encoders = {}

for col in X_encoded.columns:
    if X_encoded[col].dtype == "object" or str(X_encoded[col].dtype) == "category":
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
        label_encoders[col] = le

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y,
    test_size=0.2,       # 80% train (~504k) / 20% valid (~126k)
    random_state=42
)


In [16]:
test_encoded = test.copy()
for col in test_encoded.columns:
    if col in label_encoders:
        le = label_encoders[col]
        test_encoded[col] = le.transform(test_encoded[col].astype(str))


In [17]:
import lightgbm as lgb
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=3000,       # many trees, use early stopping
    learning_rate=0.03,      # stable for large data
    num_leaves=63,           # more capacity than 31 but safe
    max_depth=-1,            # or 12 if you see overfitting
    min_data_in_leaf=500,    # controls overfitting on big data
    min_child_samples=500,   # sklearn alias, keep in sync
    reg_lambda=1.0,
    reg_alpha=0.0,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    max_bin=255,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)

print("Best iteration:", model.best_iteration_)


[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=500 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=500 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006572 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 580
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 7
[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=500 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Info] Start training from score 62.482335
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 9.03548	valid_0's l2: 81.6399
[200]	valid_0's rmse: 8.79874	valid_0's l2: 77.4179
[300]	valid_0's rmse: 8.77762	valid_0's l2: 77.0467
[

In [18]:
# validation predictions
preds_valid = model.predict(X_valid, num_iteration=model.best_iteration_)

# test predictions for submission
preds_test = model.predict(test_encoded, num_iteration=model.best_iteration_)

submission = pd.DataFrame({
    "id": test_ids,
    "exam_score": preds_test
})

submission.to_csv("submission.csv", index=False)
submission.head()


[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=500 will be ignored. Current value: min_data_in_leaf=500
[LightGBM] [Warning] min_data_in_leaf is set=500, min_child_samples=500 will be ignored. Current value: min_data_in_leaf=500


,id,exam_score
0,630000,70.039165
1,630001,69.549832
2,630002,87.668975
3,630003,55.605406
4,630004,47.853646


In [19]:
op = {
    'id':[630000 + i for i in range(0,preds_test.shape[0])],
    "exam_score": preds_test
    
}
op= pd.DataFrame(op)
op.to_csv("submission.csv", index=False)
op.head(10)


,id,exam_score
0,630000,70.039165
1,630001,69.549832
2,630002,87.668975
3,630003,55.605406
4,630004,47.853646
5,630005,71.351171
6,630006,74.259101
7,630007,60.054294
8,630008,80.393607
9,630009,89.574887


In [20]:
op.shape

(270000, 2)